In [ ]:
# Set up environment variables
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Set up s3 vector storage through langchain:
# We will use an s3 vector bucket which allows us to use the bucket as a vector db without without managing it ourselves - AWS handles it
import boto3
client = boto3.client('s3vectors',
                       region_name="eu-central-1",
                       )
try:
    response = client.create_vector_bucket(
        vectorBucketName = 'test-bucket',
    )
    print(response)
except client.exceptions.ConflictException:
    print("The bucket exists already")


# We also need to define a vector index
try:
    response = client.create_index(
        vectorBucketName='test-bucket',
        indexName='test-index',
        dataType='float32',
        dimension=1536, # This is dependent on the embedding model we will use later
        distanceMetric='cosine',
    )
except client.exceptions.ConflictException:
    print("The index exists already")



In [ ]:
# We now want to feed chunks of our html documents into the vector bucket
# However, we are facing two problems:
# Problem 1: Our Raw html documents are noisy, we just want to extract content

# We will use Beautiful Soup to remove some noise and extract relevant content
from bs4 import BeautifulSoup
# Load our example html (TUM Examinations html page)
with open ("../../html_samples/tum_examinations.html") as f:
    html_text = f.read()
print(f"The full file has {len(html_text)} characters")

# Let's extract only the main content from the file
def extract_main_content(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    
    # Try to find the main content container
    # TUM uses id="main-content" (from the skip-nav link)
    main = (
        soup.find(id="main-content")
        or soup.find("main")
        or soup.find("article")
    )
    return str(main)

html_text = extract_main_content(html_text)

# Furthermore let's remove some noisy tags all together
def strip_nav_noise(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup.find_all(["nav", "header", "footer", "aside", "noscript", "script", "style"]):
        tag.decompose()
    return str(soup)

html_text = strip_nav_noise(html_text)
# Be aware this html cleaning is specific to this site and may be a bit crude - it just serves as an example

print(f"The stripped file has {len(html_text)} characters")

In [ ]:
# Problem 2: We need to chunk the content into smaller pieces to use them effectively in a vector db.
# Also the html text still includes a bunch of tags.
# Luckily langchain has inbuilt solutions for this:
from langchain_text_splitters import HTMLSectionSplitter, RecursiveCharacterTextSplitter

# First we use the section splitter to split the pages into broad sections (Once again this is specific to this html page!)
headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
]

html_splitter = HTMLSectionSplitter(headers_to_split_on)
# This creates a list of langchain documents, which may still be too large to upload as one
html_header_splits = html_splitter.split_text(html_text)

# Now we define chunk sizes and further split the documents into smaller chunks
chunk_size = 600
chunk_overlap = 150
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
)

# The splits should now be ready to upload
splits = text_splitter.split_documents(html_header_splits)

In [ ]:
# Now that we have our chunks ready we can upload them into out s3 vector bucket
# Langchain has inbuilt solutions to do this so we do not need to handle it manually in boto3

from langchain_aws.embeddings import BedrockEmbeddings
from langchain_aws.vectorstores.s3_vectors import AmazonS3Vectors

# Let's define an embedding model to vectorize our chunks
bedrock_client = boto3.client(
    'bedrock-runtime',
    region_name = 'eu-central-1',
    )
embedding = BedrockEmbeddings(
    client=bedrock_client,
    region_name='eu-central-1',
    model_id = "amazon.titan-embed-text-v1", # Important: This model matches the 1024 dimensions we defined for the index
    )

vector_store = AmazonS3Vectors(
    vector_bucket_name="test-bucket",
    index_name="test-index",
    embedding=embedding,
    region_name='eu-central-1',
    client=client
)

# Check if our bucket is already populated:
response = client.list_vectors(
    vectorBucketName='test-bucket',
    indexName='test-index',
    maxResults=1,
)
if not response['vectors']:
    # If the vector storage is still empty we actually add the documents
    vector_store.add_documents(
        splits
    )
    print("Added vectors succesfully")

In [ ]:
# Now that we set up out vector db we can run similarity search to retrieve information
vector_store.similarity_search_with_score(
    "Who governs examinations at TUM?"
)

In [ ]:
# Now that we have a simple RAG system let's build an agent to use it:
# First of all we need to build a tool that allows our agent to run similarity search in our s3 vector
from langchain.tools import tool

@tool
def similarity_search(text: str):
    """ Finds similar strings in a vector database.
    Args:
        text: A string to run similarity search for

    Return value:
        A list containing of dictionaries containing the search results. Each dictionary corresponds one matched
        vector and has two keys: page content for the string and similarity score for the cosine similarity (lower is better)
    """

    tuple_list = vector_store.similarity_search_with_score(text)
    
    # Reformat results to reduce noise
    res = [{'page_content': tup[0].page_content,
      'similarity_score': tup[1]}
      for tup in tuple_list]
    return res

# Another important element we need is the LLM
from langchain_aws import ChatBedrock
brt = boto3.client(service_name="bedrock-runtime",
                   region_name="eu-central-1")

model_id = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"

model = ChatBedrock(
        model_id=model_id,
        client=brt,
        model_kwargs={"temperature": 0, "top_k": 1},
    )

tools = [similarity_search]

In [ ]:
# Create the agent using LangGraph's prebuilt ReAct agent
# This sets up the full ReAct loop (LLM -> tool call -> LLM -> ... -> response) for us
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver


# MemorySaver stores conversation history in memory, keyed by thread_id
memory = MemorySaver()

agent = create_agent(
    model=model,
    tools=tools,
    checkpointer=memory,
    system_prompt="You are a helpful TUM student advisor. Use the similarity search tool to look up information before answering questions about TUM examinations and regulations.",
)

In [ ]:
# We can visualize our simple agent:
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
# Invoke the agent with a question
# The thread_id allows the agent to remember previous messages in the conversation
from langchain.messages import HumanMessage

# As long as this thread id is 1 you will continue in the same conversation/ thread
# Note: MemorySaver() handles memory in a python dictionary, so you will lose all history when the kernel restarts
config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="Who governs examinations at TUM?")]},
    config,
)

for m in response["messages"]:
    m.pretty_print()

In [ ]:
# Test multi-turn: ask a follow-up on the same thread
response = agent.invoke(
    {"messages": [HumanMessage(content="What is APSO?")]},
    config,
)

for m in response["messages"]:
    m.pretty_print()